# Notebook 3 — Train LSTM Fall Detection — IMPROVED
**Kaggle T4x2 · ~20 phút**

### Thay đổi so với v1:
- `input_size 34` → **`102`** (x,y,conf + velocity)
- `hidden 64` → **`128`** + **Bidirectional**
- Thêm **Attention layer** — model tự học frame nào quan trọng nhất
- `stride 5` → **`stride 3`** — nhiều samples hơn 40%
- `seq_len 20` → **`30`** — nhìn cả 1 giây ngã
- **Augmentation mạnh hơn**: time-warp, scale, noise
- `epochs 50` → **`80`** + LR warmup
- Export ONNX với `input_size=102`

In [ ]:
!pip install onnx onnxscript -q

In [ ]:
import os, json
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
KP_DIR = Path('/kaggle/input/datasets/nguynteincong/keypoint')

with open(KP_DIR / 'split.json') as f:
    split = json.load(f)

print(f'Train sequences: {len(split["train"])}')
print(f'Val   sequences: {len(split["val"])}')

# Verify feature size từ notebook 2
sample_kps = np.load(KP_DIR / f'{split["train"][0]}_kps.npy')
print(f'Feature size: {sample_kps.shape[1]} (expected 102 = 51 kp + 51 velocity)')

In [ ]:
# ── THAY ĐỔI: seq_len=30, stride=3, feature=102 ──────────────────────────────
SEQ_LEN    = 30   # tăng từ 20 → 30 (~1 giây ở 30fps)
STRIDE     = 3    # giảm từ 5 → 3 → nhiều samples hơn ~67%
FEAT_SIZE  = 102  # tăng từ 34 → 102

class FallWindowDataset(Dataset):
    def __init__(self, seq_names, kp_dir, seq_len=30, stride=3, augment=False):
        self.samples = []
        self.augment = augment

        for name in seq_names:
            kps_path = kp_dir / f'{name}_kps.npy'
            lbl_path = kp_dir / f'{name}_labels.npy'
            if not kps_path.exists():
                continue

            kps    = np.load(kps_path)    # (T, 102)
            labels = np.load(lbl_path)    # (T,)
            T      = len(kps)
            if T < seq_len:
                continue

            for start in range(0, T - seq_len + 1, stride):
                window = kps[start:start + seq_len]
                label  = int(labels[start:start + seq_len].mean() > 0.5)
                self.samples.append((window.astype(np.float32), label))

        n_fall   = sum(1 for _, l in self.samples if l == 1)
        n_normal = len(self.samples) - n_fall
        print(f'  {"aug" if augment else "val":3s}: {len(self.samples):5d} samples '
              f'(fall={n_fall}, normal={n_normal})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        kps, label = self.samples[idx]

        if self.augment:
            # 1. Flip ngang: đảo x-coord (index 0,3,6,...) trong 51 đầu
            if np.random.rand() < 0.5:
                kps = kps.copy()
                # x nằm ở index 0,3,6,...,48 (bước 3 trong 51 kp đầu)
                kps[:, 0:51:3] = 1.0 - kps[:, 0:51:3]
                # velocity x cũng đảo dấu
                kps[:, 51::3]  = -kps[:, 51::3]

            # 2. Gaussian noise nhỏ vào tọa độ
            noise = np.random.randn(*kps.shape).astype(np.float32) * 0.008
            kps   = kps + noise

            # 3. Scale ngẫu nhiên (mô phỏng camera xa/gần)
            scale = np.random.uniform(0.85, 1.15)
            kps[:, :51] = kps[:, :51] * scale

            # 4. Time mask: zero-out 2-4 frame ngẫu nhiên (mô phỏng occlusion)
            if np.random.rand() < 0.3:
                t_start = np.random.randint(0, len(kps) - 3)
                t_len   = np.random.randint(2, 5)
                kps[t_start:t_start+t_len] = 0.0

        return torch.tensor(kps), torch.tensor(label, dtype=torch.long)


print('Tạo dataset...')
train_ds = FallWindowDataset(split['train'], KP_DIR, SEQ_LEN, STRIDE, augment=True)
val_ds   = FallWindowDataset(split['val'],   KP_DIR, SEQ_LEN, STRIDE, augment=False)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ── THAY ĐỔI CHÍNH: Bidirectional LSTM + Attention ───────────────────────────

class TemporalAttention(nn.Module):
    """Học xem frame nào trong sequence quan trọng nhất."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out):
        # lstm_out: (B, T, hidden_dim)
        scores  = self.attn(lstm_out).squeeze(-1)   # (B, T)
        weights = F.softmax(scores, dim=1)          # (B, T)
        context = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)  # (B, hidden_dim)
        return context


class FallLSTM(nn.Module):
    def __init__(
        self,
        input_size = 102,   # tăng từ 34 → 102
        hidden     = 128,   # tăng từ 64 → 128
        n_layers   = 2,
        dropout    = 0.3
    ):
        super().__init__()

        # ── Bidirectional LSTM ────────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size,
            hidden,
            num_layers   = n_layers,
            batch_first  = True,
            dropout      = dropout if n_layers > 1 else 0.0,
            bidirectional= True,    # forward + backward → hidden*2
        )

        # ── Attention ─────────────────────────────────────────────────────────
        self.attention = TemporalAttention(hidden * 2)

        # ── Classifier head ───────────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        # x: (B, T, 102)
        lstm_out, _ = self.lstm(x)          # (B, T, hidden*2)
        context     = self.attention(lstm_out)  # (B, hidden*2)
        return self.head(context)


model   = FallLSTM().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}  (~{n_params*4/1024:.0f} KB)')
print(f'Input:  (batch, {SEQ_LEN}, {FEAT_SIZE})')
print(f'Output: (batch, 2)')

In [ ]:
# Class weight để xử lý imbalance
labels_all = [l for _, l in train_ds.samples]
n_normal   = labels_all.count(0)
n_fall     = labels_all.count(1)
total      = len(labels_all)

w_normal = total / (2 * n_normal)
w_fall   = total / (2 * n_fall)
weights  = torch.tensor([w_normal, w_fall], dtype=torch.float32).to(DEVICE)

print(f'Normal: {n_normal}  Fall: {n_fall}')
print(f'Class weights → normal={w_normal:.2f}  fall={w_fall:.2f}')

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Warmup 5 epoch, rồi cosine decay
def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    return 0.5 * (1 + np.cos(np.pi * (epoch - warmup) / (80 - warmup)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
EPOCHS    = 80    # tăng từ 50 → 80
best_f1   = 0.0
best_path = '/kaggle/working/fall_lstm_best.pt'


def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
                optimizer.step()
            total_loss += loss.item() * len(y)
            preds       = logits.argmax(1)
            correct    += (preds == y).sum().item()
            total      += len(y)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    return total_loss / total, correct / total, f1, all_preds, all_labels


print(f'{"Epoch":>5}  {"TrainLoss":>9}  {"TrainAcc":>8}  {"ValLoss":>7}  {"ValAcc":>6}  {"F1-Fall":>7}  {"LR":>8}')
print('─' * 70)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, _,      _, _ = run_epoch(model, train_loader, train=True)
    va_loss, va_acc, va_f1, preds, labels = run_epoch(model, val_loader, train=False)
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    flag = ''
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save(model.state_dict(), best_path)
        flag = ' ← best'

    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:>5}  {tr_loss:>9.4f}  {tr_acc:>8.3f}  '
              f'{va_loss:>7.4f}  {va_acc:>6.3f}  {va_f1:>7.4f}  {current_lr:>8.6f}{flag}')

print(f'\nBest F1-Fall: {best_f1:.4f}')

In [ ]:
# Báo cáo chi tiết
model.load_state_dict(torch.load(best_path))
_, _, _, preds, labels = run_epoch(model, val_loader, train=False)

print('Classification Report:')
print(classification_report(labels, preds, target_names=['Normal', 'Fall']))

print('Confusion Matrix (rows=actual, cols=predicted):')
cm = confusion_matrix(labels, preds)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f'\nFalse Positive Rate: {fp/(fp+tn):.3f}  (báo động giả)')
print(f'False Negative Rate: {fn/(fn+tp):.3f}  (bỏ sót ngã — nguy hiểm!)')

In [ ]:
# Export ONNX — input_size phải khớp với 102
model.eval()
dummy     = torch.zeros(1, SEQ_LEN, FEAT_SIZE).to(DEVICE)
onnx_path = '/kaggle/working/fall_lstm.onnx'

torch.onnx.export(
    model, dummy, onnx_path,
    input_names  = ['keypoints'],
    output_names = ['logits'],
    dynamic_axes = {'keypoints': {0: 'batch'}},
    opset_version= 12,
)

print(f'ONNX saved: {onnx_path}')
print(f'Size: {os.path.getsize(onnx_path) / 1024:.1f} KB')
print(f'Input shape: (batch, {SEQ_LEN}, {FEAT_SIZE})')

In [ ]:
print('=== Files sẵn sàng download ===')
outputs = [
    ('/kaggle/working/fall_lstm_best.pt', 'PyTorch weights — dùng trong testt.py'),
    ('/kaggle/working/fall_lstm.onnx',    'ONNX — convert TensorRT trên Jetson'),
]
for path, desc in outputs:
    size = os.path.getsize(path) / 1024
    print(f'  {desc:<50} {size:6.1f} KB  {path}')

print()
print('=== Cập nhật testt.py sau khi dùng model mới ===')
print('Đổi 3 chỗ trong testt_fixed.py:')
print("  1. FallLSTM(input_size=102, hidden=128)  # từ 34 → 102")
print("  2. Trong get_keypoints(): lấy thêm conf, trả về 51 features")
print("  3. Trong detect_fall(): concat velocity trước khi inference")
print("  4. --seq_len 30  (thay vì 20)")